# Phi-4 Zero-Shot Hallucination Verifier — Fixed Notebook

This version fixes the main evaluation problem in the old notebook:

- The task is now **hallucination detection**, not open-ended QA scoring.
- Each original row becomes two labeled samples:
  - `SUPPORTED = 1` using `right_answer`
  - `HALLUCINATED = 0` using `hallucinated_answer`
- Metrics are calculated against real labels using Accuracy / Precision / Recall / F1 / Confusion Matrix.
- The summarized dataset is aligned with the original dataset by question and answer fields.
- Bad summarized rows can be filtered or safely replaced with original/selected evidence.


In [2]:
# GPU-safe setup for Kaggle / Lightning
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
print("CUDA devices visible:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA devices visible: 1
GPU: Tesla T4


In [3]:
# Install requirements
# Run once, then restart the session if packages were changed.
import subprocess, sys

packages = [
    'transformers==4.52.4',
    'accelerate==1.7.0',
    'bitsandbytes==0.46.0',
    'pandas>=2.0.0',
    'numpy>=1.24.0',
    'scikit-learn>=1.3.0',
    'scipy>=1.11.0',
    'tqdm>=4.66.0',
    'tabulate>=0.9.0',
]

for pkg in packages:
    print('Installing:', pkg)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('Done. If packages were installed/updated, restart session then Run All.')


Installing: transformers==4.52.4
Installing: accelerate==1.7.0
Installing: bitsandbytes==0.46.0
Installing: pandas>=2.0.0
Installing: numpy>=1.24.0
Installing: scikit-learn>=1.3.0
Installing: scipy>=1.11.0
Installing: tqdm>=4.66.0
Installing: tabulate>=0.9.0
Done. If packages were installed/updated, restart session then Run All.


In [4]:
import os
import json
import random
import re
import time
import gc
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm
from tabulate import tabulate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    matthews_corrcoef,
)
from scipy import stats

warnings.filterwarnings('ignore')

# ─── Global Configuration ────────────────────────────────────────────────────
SEED              = 42
SAMPLE_RECORDS    = 10000      # number of original QA records before doubling. Use None for full 10k.
TEST_RATIO        = 0.20
MODEL_ID          = 'microsoft/Phi-4-mini-instruct'
USE_4BIT          = torch.cuda.is_available()
BATCH_SIZE        = 4         # keep 1 on T4 if memory is tight
MAX_INPUT_TOKENS  = 2048
MAX_NEW_TOKENS    = 4        # only needs Supported/Hallucinated

# Important: choose summarized context behavior
# 'safe'   = use summary only when answer_preserved_in_final_summary=True, otherwise fallback to selected_evidence/original_knowledge
# 'strict' = drop rows where summarized answer preservation failed
# 'raw'    = use summarized knowledge/abstractive_summary directly, no fallback/filter
SUMMARY_MODE = 'safe'

INPUT_DIR  = Path('/kaggle/input')
WORK_DIR   = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
OUTPUT_DIR = WORK_DIR / 'phi4_zero_shot_verifier_fixed_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Works on Kaggle, Lightning, and local uploaded files
ORIG_CANDIDATES = [
    Path('/kaggle/input/datasets/ahmedahmedelhayes/qaqaqooo/qa_data.json'),

]
SUMM_CANDIDATES = [
    Path('/kaggle/input/datasets/ahmedahmedelhayes/ssmmmm/qa_data_answer_focused_summarized.json'),
]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'Device       : {"cuda" if torch.cuda.is_available() else "cpu"}')
print(f'Use 4-bit    : {USE_4BIT}')
print(f'Model        : {MODEL_ID}')
print(f'Sample records: {SAMPLE_RECORDS}')
print(f'Summary mode : {SUMMARY_MODE}')
print(f'Output dir   : {OUTPUT_DIR}')


Device       : cuda
Use 4-bit    : True
Model        : microsoft/Phi-4-mini-instruct
Sample records: 10000
Summary mode : safe
Output dir   : /kaggle/working/phi4_zero_shot_verifier_fixed_outputs


In [5]:
def resolve_dataset_path(candidates, filename, keywords=()):
    for p in candidates:
        if p.exists():
            return p

    matches = []
    if INPUT_DIR.exists():
        for p in INPUT_DIR.rglob(filename):
            s = str(p).lower()
            score = sum(k.lower() in s for k in keywords)
            matches.append((score, p))

    if matches:
        matches.sort(key=lambda x: (-x[0], len(str(x[1]))))
        return matches[0][1]

    raise FileNotFoundError(f'Could not find {filename}. Attach the dataset first.')


def load_json_or_jsonl(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        if isinstance(data, list):
            return pd.DataFrame(data)
        if isinstance(data, dict):
            return pd.DataFrame([data])
        return pd.DataFrame(data)
    except json.JSONDecodeError:
        rows = []
        with open(path, 'r', encoding='utf-8') as f:
            for line_no, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as e:
                    raise ValueError(f'Invalid JSONL at line {line_no}: {e}')
        return pd.DataFrame(rows)

ORIG_PATH = resolve_dataset_path(ORIG_CANDIDATES, 'qa_data.json', ('original', 'qa_data'))
SUMM_PATH = resolve_dataset_path(SUMM_CANDIDATES, 'qa_data_answer_focused_summarized.json', ('summ', 'summary'))

df_orig_raw = load_json_or_jsonl(ORIG_PATH)
df_summ_raw = load_json_or_jsonl(SUMM_PATH)

print('Original path:', ORIG_PATH)
print('Summarized path:', SUMM_PATH)
print('Original shape:', df_orig_raw.shape)
print('Summarized shape:', df_summ_raw.shape)
print('\nOriginal columns:', list(df_orig_raw.columns))
print('\nSummarized columns:', list(df_summ_raw.columns))


Original path: /kaggle/input/datasets/ahmedahmedelhayes/qaqaqooo/qa_data.json
Summarized path: /kaggle/input/datasets/ahmedahmedelhayes/ssmmmm/qa_data_answer_focused_summarized.json
Original shape: (10000, 4)
Summarized shape: (10000, 17)

Original columns: ['knowledge', 'question', 'right_answer', 'hallucinated_answer']

Summarized columns: ['knowledge', 'question', 'right_answer', 'hallucinated_answer', 'original_knowledge', 'selected_evidence', 'abstractive_summary', 'summary_method', 'selection_fallback_to_original', 'summary_fallback_to_selected_evidence', 'answer_preserved_in_final_summary', 'original_char_len', 'selected_char_len', 'summary_char_len', 'num_original_sentences', 'num_selected_sentences', 'selected_answer_preserved']


In [6]:
# ─── Dataset audit: labels and quality ───────────────────────────────────────
REQUIRED = ['knowledge', 'question', 'right_answer', 'hallucinated_answer']
for name, df in [('Original', df_orig_raw), ('Summarized', df_summ_raw)]:
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise ValueError(f'{name} missing columns: {missing}')

print('Original QA records:', len(df_orig_raw))
print('Summarized QA records:', len(df_summ_raw))
print('Binary samples after expansion should be:', len(df_orig_raw) * 2)
print('Labels after expansion: SUPPORTED=1 and HALLUCINATED=0, balanced by design.')

# Basic duplicate / empty checks
def audit_df(df, name):
    print(f'\n{name} audit')
    print('- empty question:', int(df['question'].fillna('').str.strip().eq('').sum()))
    print('- empty right_answer:', int(df['right_answer'].fillna('').str.strip().eq('').sum()))
    print('- empty hallucinated_answer:', int(df['hallucinated_answer'].fillna('').str.strip().eq('').sum()))
    print('- duplicate questions:', int(df.duplicated('question').sum()))
    same_ans = (df['right_answer'].fillna('').str.strip().str.lower() ==
                df['hallucinated_answer'].fillna('').str.strip().str.lower()).sum()
    print('- right_answer == hallucinated_answer:', int(same_ans))

audit_df(df_orig_raw, 'Original')
audit_df(df_summ_raw, 'Summarized')

if 'answer_preserved_in_final_summary' in df_summ_raw.columns:
    print('\nSummary quality metadata:')
    for col in ['answer_preserved_in_final_summary', 'selected_answer_preserved', 'summary_fallback_to_selected_evidence', 'selection_fallback_to_original']:
        if col in df_summ_raw.columns:
            print(col, df_summ_raw[col].value_counts(dropna=False).to_dict())


Original QA records: 10000
Summarized QA records: 10000
Binary samples after expansion should be: 20000
Labels after expansion: SUPPORTED=1 and HALLUCINATED=0, balanced by design.

Original audit
- empty question: 0
- empty right_answer: 0
- empty hallucinated_answer: 0
- duplicate questions: 0
- right_answer == hallucinated_answer: 0

Summarized audit
- empty question: 0
- empty right_answer: 0
- empty hallucinated_answer: 0
- duplicate questions: 0
- right_answer == hallucinated_answer: 0

Summary quality metadata:
answer_preserved_in_final_summary {True: 9568, False: 432}
selected_answer_preserved {True: 9536, False: 464}
summary_fallback_to_selected_evidence {False: 7134, True: 2866}
selection_fallback_to_original {False: 9536, True: 464}


In [7]:
# ─── Clean and align original/summarized datasets ────────────────────────────
def norm_key(s):
    return re.sub(r'\s+', ' ', str(s).strip().lower())

for df in [df_orig_raw, df_summ_raw]:
    for c in REQUIRED:
        df[c] = df[c].fillna('').astype(str).str.strip()
    df['_key'] = (
        df['question'].map(norm_key) + ' || ' +
        df['right_answer'].map(norm_key) + ' || ' +
        df['hallucinated_answer'].map(norm_key)
    )

# Drop broken rows before alignment
bad_orig = (
    df_orig_raw['question'].eq('') |
    df_orig_raw['right_answer'].eq('') |
    df_orig_raw['hallucinated_answer'].eq('') |
    (df_orig_raw['right_answer'].map(norm_key) == df_orig_raw['hallucinated_answer'].map(norm_key))
)
bad_summ = (
    df_summ_raw['question'].eq('') |
    df_summ_raw['right_answer'].eq('') |
    df_summ_raw['hallucinated_answer'].eq('') |
    (df_summ_raw['right_answer'].map(norm_key) == df_summ_raw['hallucinated_answer'].map(norm_key))
)

df_orig = df_orig_raw.loc[~bad_orig].copy().reset_index(drop=True)
df_summ = df_summ_raw.loc[~bad_summ].copy().reset_index(drop=True)

# Align on question + answers to guarantee same records in both experiments
common_keys = sorted(set(df_orig['_key']) & set(df_summ['_key']))
print(f'Common aligned records: {len(common_keys):,}')
print(f'Original-only records : {len(set(df_orig["_key"]) - set(df_summ["_key"])):,}')
print(f'Summary-only records  : {len(set(df_summ["_key"]) - set(df_orig["_key"])):,}')

if len(common_keys) == 0:
    raise ValueError('No aligned records found. Check that both datasets belong to the same source.')

df_orig = df_orig.set_index('_key').loc[common_keys].reset_index()
df_summ = df_summ.set_index('_key').loc[common_keys].reset_index()

# Optional strict filter for bad summaries
if SUMMARY_MODE == 'strict' and 'answer_preserved_in_final_summary' in df_summ.columns:
    keep = df_summ['answer_preserved_in_final_summary'].astype(bool).values
    df_orig = df_orig.loc[keep].reset_index(drop=True)
    df_summ = df_summ.loc[keep].reset_index(drop=True)
    print(f'After strict answer-preserved filter: {len(df_orig):,} records')

print('Final aligned original shape:', df_orig.shape)
print('Final aligned summarized shape:', df_summ.shape)


Common aligned records: 10,000
Original-only records : 0
Summary-only records  : 0
Final aligned original shape: (10000, 5)
Final aligned summarized shape: (10000, 18)


In [8]:
# ─── Better summarized context: evidence + summary ──────────────────────────
def clean_text(x):
    return re.sub(r'\s+', ' ', str(x).strip())

def get_original_context(row):
    return clean_text(row.get('knowledge', ''))

def get_summarized_context(row):
    summary = clean_text(row.get('abstractive_summary', ''))
    selected = clean_text(row.get('selected_evidence', ''))
    original = clean_text(row.get('original_knowledge', row.get('knowledge', '')))

    preserved = bool(row.get('answer_preserved_in_final_summary', True))

    # Best case: use evidence + summary
    if selected and summary:
        return selected + "\n\nSummary:\n" + summary

    # If summary lost the answer, don't trust it alone
    if not preserved:
        if selected:
            return selected
        if original:
            return original

    if summary:
        return summary
    if selected:
        return selected
    return original

df_orig['_context'] = df_orig.apply(get_original_context, axis=1)
df_summ['_context'] = df_summ.apply(get_summarized_context, axis=1)

print('Context length stats:')
ctx_stats = pd.DataFrame({
    'original_chars': df_orig['_context'].str.len(),
    'summarized_chars': df_summ['_context'].str.len(),
})
print(ctx_stats.describe().round(1))

Context length stats:
       original_chars  summarized_chars
count         10000.0           10000.0
mean            343.9             455.8
std             134.7             151.3
min              75.0             124.0
25%             246.0             345.0
50%             321.0             432.0
75%             416.0             543.0
max            1557.0            1582.0


In [9]:
# ─── Convert QA records into binary hallucination-verification samples ───────
def build_binary_dataset(df_records, source_name):
    rows = []
    for rec_id, r in df_records.reset_index(drop=True).iterrows():
        base = {
            'record_id': rec_id,
            'source': source_name,
            'context': str(r['_context']).strip(),
            'question': str(r['question']).strip(),
            'gold_answer': str(r['right_answer']).strip(),
        }
        rows.append({
            **base,
            'answer_to_check': str(r['right_answer']).strip(),
            'label': 1,
            'label_name': 'SUPPORTED',
        })
        rows.append({
            **base,
            'answer_to_check': str(r['hallucinated_answer']).strip(),
            'label': 0,
            'label_name': 'HALLUCINATED',
        })
    return pd.DataFrame(rows)

bin_orig = build_binary_dataset(df_orig, 'original')
bin_summ = build_binary_dataset(df_summ, 'summarized')

print('Original binary shape:', bin_orig.shape)
print('Summarized binary shape:', bin_summ.shape)
print('\nLabel distribution:')
print(bin_orig['label_name'].value_counts())
print(bin_summ['label_name'].value_counts())

# Split by record_id, not by binary row, to avoid leakage between supported/hallucinated pair.
record_ids = np.arange(len(df_orig))
if SAMPLE_RECORDS is not None and SAMPLE_RECORDS < len(record_ids):
    rng = np.random.default_rng(SEED)
    record_ids = np.sort(rng.choice(record_ids, size=SAMPLE_RECORDS, replace=False))

_, test_record_ids = train_test_split(record_ids, test_size=TEST_RATIO, random_state=SEED)
test_record_ids = set(test_record_ids.tolist())

test_orig = bin_orig[bin_orig['record_id'].isin(test_record_ids)].reset_index(drop=True)
test_summ = bin_summ[bin_summ['record_id'].isin(test_record_ids)].reset_index(drop=True)

print(f'Test records: {len(test_record_ids):,}')
print(f'Test binary samples per method: {len(test_orig):,}')
print(test_orig['label_name'].value_counts())


Original binary shape: (20000, 8)
Summarized binary shape: (20000, 8)

Label distribution:
label_name
SUPPORTED       10000
HALLUCINATED    10000
Name: count, dtype: int64
label_name
SUPPORTED       10000
HALLUCINATED    10000
Name: count, dtype: int64
Test records: 2,000
Test binary samples per method: 4,000
label_name
SUPPORTED       2000
HALLUCINATED    2000
Name: count, dtype: int64


In [10]:
# ─── Load Phi-4-mini ─────────────────────────────────────────────────────────
print(f'Loading tokenizer: {MODEL_ID}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

bnb_cfg = None
if USE_4BIT:
    compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
    print('4-bit enabled:', compute_dtype)

print(f'Loading model: {MODEL_ID}')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map={"": 0} if USE_4BIT else None,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=False,
    attn_implementation='eager',
)
model.eval()
print('Model loaded.')


Loading tokenizer: microsoft/Phi-4-mini-instruct
4-bit enabled: torch.bfloat16
Loading model: microsoft/Phi-4-mini-instruct


2026-06-12 19:15:07.070812: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781291707.094570     282 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781291707.101693     282 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781291707.122343     282 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781291707.122362     282 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781291707.122364     282 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded.


In [11]:
SYSTEM_PROMPT = """You are a strict but fair hallucination verifier.

Decide whether the Candidate Answer is supported by the Context for the Question.

Return exactly one word only:
SUPPORTED
or
HALLUCINATED

Rules:
- SUPPORTED if the context contains the same answer, equivalent wording, abbreviation, or paraphrase.
- SUPPORTED if the answer is directly implied by the context.
- HALLUCINATED if the answer contradicts the context.
- HALLUCINATED if the answer adds important details not found in the context.
- Do not require exact word matching.
- Do not explain.
"""
def build_verifier_prompt(context, question, answer):
    return f"""<|system|>
{SYSTEM_PROMPT}
<|end|>
<|user|>
Context:
{str(context).strip()}

Question:
{str(question).strip()}

Candidate Answer:
{str(answer).strip()}
<|end|>
<|assistant|>
"""

GEN_CONFIG = dict(
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,
    temperature=1.0,
    repetition_penalty=1.0,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

def extract_generation(full_text, prompt):
    if '<|assistant|>' in full_text:
        out = full_text.split('<|assistant|>')[-1]
    elif full_text.startswith(prompt):
        out = full_text[len(prompt):]
    else:
        out = full_text
    out = out.replace('<|end|>', '').replace('<|endoftext|>', '').strip()
    return out.splitlines()[0].strip() if out else ''

def parse_label(text):
    t = str(text).strip().upper()
    # Prefer exact first token, because Phi may output punctuation.
    first = re.sub(r'[^A-Z]', '', t.split()[0]) if t.split() else ''
    if first.startswith('SUPPORTED'):
        return 1
    if first.startswith('HALLUCINATED'):
        return 0
    if 'SUPPORTED' in t and 'HALLUCINATED' not in t:
        return 1
    if 'HALLUCINATED' in t:
        return 0
    # Conservative fallback: unknown verifier outputs are counted as hallucinated.
    return 0

print('Verifier prompt and parser ready.')


Verifier prompt and parser ready.


In [12]:
def run_verifier(df, desc='verifier'):
    raw_outputs = []
    pred_labels = []
    prompts = []

    rows = list(df.itertuples(index=False))
    for start in tqdm(range(0, len(rows), BATCH_SIZE), desc=desc):
        batch = rows[start:start+BATCH_SIZE]
        batch_prompts = [
            build_verifier_prompt(r.context, r.question, r.answer_to_check)
            for r in batch
        ]
        enc = tokenizer(
            batch_prompts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_TOKENS,
        ).to(model.device)

        with torch.no_grad():
            out = model.generate(**enc, **GEN_CONFIG)

        for i, ids in enumerate(out):
            full = tokenizer.decode(ids, skip_special_tokens=False)
            raw = extract_generation(full, batch_prompts[i])
            raw_outputs.append(raw)
            pred_labels.append(parse_label(raw))
            prompts.append(batch_prompts[i])

        del enc, out
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    result = df.copy()
    result['raw_model_output'] = raw_outputs
    result['pred_label'] = pred_labels
    result['pred_label_name'] = np.where(result['pred_label'].eq(1), 'SUPPORTED', 'HALLUCINATED')
    result['correct'] = result['pred_label'].eq(result['label']).astype(int)
    return result


In [13]:
print('='*70)
print('RUNNING ZERO-SHOT VERIFIER: ORIGINAL CONTEXT')
print('='*70)
t0 = time.time()
pred_orig = run_verifier(test_orig, desc='Original verifier')
time_orig = time.time() - t0
print(f'Original verifier time: {time_orig:.1f}s')

print('\nExamples:')
print(pred_orig[['question', 'answer_to_check', 'label_name', 'raw_model_output', 'pred_label_name', 'correct']].head(6).to_string(index=False))


RUNNING ZERO-SHOT VERIFIER: ORIGINAL CONTEXT


Original verifier: 100%|██████████| 1000/1000 [1:14:27<00:00,  4.47s/it]

Original verifier time: 4467.9s

Examples:
                                                                                                    question                                                                                                 answer_to_check   label_name raw_model_output pred_label_name  correct
           "(I'm 10,000 Years Old)" was the subtitle of the 11th album by which iconic rock and roll singer?                                                                                                   Elvis Presley    SUPPORTED        SUPPORTED       SUPPORTED        1
           "(I'm 10,000 Years Old)" was the subtitle of the 11th album by which iconic rock and roll singer?                                                      "The King" released a rock and roll album with this title. HALLUCINATED        SUPPORTED       SUPPORTED        0
"A Rickle in Time" is the first episode in which season, of the animated television series "Rick and Morty"?                     

In [14]:
print('='*70)
print('RUNNING ZERO-SHOT VERIFIER: SUMMARIZED CONTEXT')
print('='*70)
t0 = time.time()
pred_summ = run_verifier(test_summ, desc='Summarized verifier')
time_summ = time.time() - t0
print(f'Summarized verifier time: {time_summ:.1f}s')

print('\nExamples:')
print(pred_summ[['question', 'answer_to_check', 'label_name', 'raw_model_output', 'pred_label_name', 'correct']].head(6).to_string(index=False))


RUNNING ZERO-SHOT VERIFIER: SUMMARIZED CONTEXT


Summarized verifier: 100%|██████████| 1000/1000 [1:20:26<00:00,  4.83s/it]

Summarized verifier time: 4826.2s

Examples:
                                                                                                    question                                                                                                 answer_to_check   label_name raw_model_output pred_label_name  correct
           "(I'm 10,000 Years Old)" was the subtitle of the 11th album by which iconic rock and roll singer?                                                                                                   Elvis Presley    SUPPORTED        SUPPORTED       SUPPORTED        1
           "(I'm 10,000 Years Old)" was the subtitle of the 11th album by which iconic rock and roll singer?                                                      "The King" released a rock and roll album with this title. HALLUCINATED        SUPPORTED       SUPPORTED        0
"A Rickle in Time" is the first episode in which season, of the animated television series "Rick and Morty"?                   

In [15]:
# ─── Proper binary classification metrics ───────────────────────────────────
def compute_binary_metrics(df):
    y_true = df['label'].astype(int).values
    y_pred = df['pred_label'].astype(int).values
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', pos_label=1, zero_division=0
    )
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'supported_precision': precision,
        'supported_recall': recall,
        'supported_f1': f1,
        'macro_precision': macro_p,
        'macro_recall': macro_r,
        'macro_f1': macro_f1,
        'mcc': matthews_corrcoef(y_true, y_pred),
        'tn_hallucinated_as_hallucinated': int(cm[0,0]),
        'fp_hallucinated_as_supported': int(cm[0,1]),
        'fn_supported_as_hallucinated': int(cm[1,0]),
        'tp_supported_as_supported': int(cm[1,1]),
    }

metrics_orig = compute_binary_metrics(pred_orig)
metrics_summ = compute_binary_metrics(pred_summ)

compare = pd.DataFrame([
    {'metric': k, 'original': metrics_orig[k], 'summarized': metrics_summ[k], 'summ_minus_orig': metrics_summ[k] - metrics_orig[k]}
    for k in metrics_orig.keys()
])

print('\nOriginal classification report:')
print(classification_report(pred_orig['label'], pred_orig['pred_label'], target_names=['HALLUCINATED', 'SUPPORTED'], zero_division=0))

print('\nSummarized classification report:')
print(classification_report(pred_summ['label'], pred_summ['pred_label'], target_names=['HALLUCINATED', 'SUPPORTED'], zero_division=0))

print('\nComparison:')
print(tabulate(compare.round(4), headers='keys', tablefmt='grid', showindex=False))



Original classification report:
              precision    recall  f1-score   support

HALLUCINATED       0.92      0.64      0.76      2000
   SUPPORTED       0.73      0.95      0.82      2000

    accuracy                           0.79      4000
   macro avg       0.82      0.79      0.79      4000
weighted avg       0.82      0.79      0.79      4000


Summarized classification report:
              precision    recall  f1-score   support

HALLUCINATED       0.94      0.71      0.80      2000
   SUPPORTED       0.76      0.95      0.85      2000

    accuracy                           0.83      4000
   macro avg       0.85      0.83      0.83      4000
weighted avg       0.85      0.83      0.83      4000


Comparison:
+---------------------------------+------------+--------------+-------------------+
| metric                          |   original |   summarized |   summ_minus_orig |
+=================================+============+==============+===================+
| accuracy   

In [16]:
# ─── Paired significance test on correctness ─────────────────────────────────
# Because original and summarized use the same binary samples, compare correctness per sample.
orig_correct = pred_orig['correct'].values
summ_correct = pred_summ['correct'].values

# McNemar-style counts
b = int(((orig_correct == 1) & (summ_correct == 0)).sum())  # original correct, summarized wrong
c = int(((orig_correct == 0) & (summ_correct == 1)).sum())  # original wrong, summarized correct
mcnemar_chi2 = ((abs(b - c) - 1) ** 2 / (b + c)) if (b + c) > 0 else 0.0
mcnemar_p = float(stats.chi2.sf(mcnemar_chi2, 1)) if (b + c) > 0 else 1.0

print('Paired correctness comparison:')
print(f'Original correct, summarized wrong: {b}')
print(f'Original wrong, summarized correct: {c}')
print(f'McNemar chi-square: {mcnemar_chi2:.4f}')
print(f'p-value: {mcnemar_p:.4f}')

# Inspect disagreement cases
merged = pred_orig[['record_id','answer_to_check','label','label_name','pred_label','correct']].rename(
    columns={'pred_label':'pred_orig_label','correct':'correct_orig'}
).merge(
    pred_summ[['record_id','answer_to_check','pred_label','correct']].rename(
        columns={'pred_label':'pred_summ_label','correct':'correct_summ'}
    ),
    on=['record_id','answer_to_check'],
    how='inner'
)

disagree = merged[merged['pred_orig_label'] != merged['pred_summ_label']].copy()
print(f'Disagreement cases: {len(disagree)} / {len(merged)}')


Paired correctness comparison:
Original correct, summarized wrong: 68
Original wrong, summarized correct: 208
McNemar chi-square: 70.0036
p-value: 0.0000
Disagreement cases: 276 / 3998


In [17]:
# ─── Save outputs ────────────────────────────────────────────────────────────
pred_orig.to_csv(OUTPUT_DIR / 'phi4_verifier_predictions_original.csv', index=False)
pred_summ.to_csv(OUTPUT_DIR / 'phi4_verifier_predictions_summarized.csv', index=False)
compare.to_csv(OUTPUT_DIR / 'phi4_verifier_metrics_comparison.csv', index=False)
disagree.to_csv(OUTPUT_DIR / 'phi4_verifier_disagreements.csv', index=False)

summary_text = f"""
PHI-4 ZERO-SHOT HALLUCINATION VERIFIER — FIXED RUN SUMMARY
===========================================================

Task:
Binary hallucination detection over (context, question, candidate_answer).
Labels:
SUPPORTED=1 for right_answer.
HALLUCINATED=0 for hallucinated_answer.

Aligned QA records used before test split: {len(df_orig):,}
Test QA records: {len(test_record_ids):,}
Test binary samples per method: {len(test_orig):,}
Summary mode: {SUMMARY_MODE}
Model: {MODEL_ID}

Original metrics:
{pd.Series(metrics_orig).round(4).to_string()}

Summarized metrics:
{pd.Series(metrics_summ).round(4).to_string()}

McNemar paired correctness test:
Original correct / summarized wrong: {b}
Original wrong / summarized correct: {c}
chi-square: {mcnemar_chi2:.4f}
p-value: {mcnemar_p:.4f}

Main fix:
The previous notebook evaluated generated answers against gold answers using fuzzy/semantic scores.
This can produce misleadingly high numbers and does not directly evaluate hallucination detection.
This notebook evaluates real binary labels built from right_answer and hallucinated_answer.
"""

with open(OUTPUT_DIR / 'phi4_verifier_final_summary.txt', 'w', encoding='utf-8') as f:
    f.write(summary_text)

print(summary_text)
print('\nSaved files:')
for p in sorted(OUTPUT_DIR.glob('*')):
    print(p.name, p.stat().st_size, 'bytes')



PHI-4 ZERO-SHOT HALLUCINATION VERIFIER — FIXED RUN SUMMARY

Task:
Binary hallucination detection over (context, question, candidate_answer).
Labels:
SUPPORTED=1 for right_answer.
HALLUCINATED=0 for hallucinated_answer.

Aligned QA records used before test split: 10,000
Test QA records: 2,000
Test binary samples per method: 4,000
Summary mode: safe
Model: microsoft/Phi-4-mini-instruct

Original metrics:
accuracy                              0.7938
supported_precision                   0.7254
supported_recall                      0.9455
supported_f1                          0.8209
macro_precision                       0.8236
macro_recall                          0.7938
macro_f1                              0.7889
mcc                                   0.6166
tn_hallucinated_as_hallucinated    1284.0000
fp_hallucinated_as_supported        716.0000
fn_supported_as_hallucinated        109.0000
tp_supported_as_supported          1891.0000

Summarized metrics:
accuracy                        

In [18]:
# Optional: zip all output CSV/TXT files for download from Kaggle/Lightning
import shutil
zip_base = str(OUTPUT_DIR.parent / 'phi4_zero_shot_verifier_fixed_outputs')
zip_path = shutil.make_archive(zip_base, 'zip', OUTPUT_DIR)
print('ZIP ready:', zip_path)


ZIP ready: /kaggle/working/phi4_zero_shot_verifier_fixed_outputs.zip


In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/ALL_OUTPUTS",
    "zip",
    "/kaggle/working"
)

print("ZIP READY: /kaggle/working/ALL_OUTPUTS.zip")